In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/towards-ds-20records/towards_data_science_100_articles.csv


In [2]:
# Optional in Kaggle, since transformers & torch are preinstalled, but include if running elsewhere
# !pip install transformers tqdm

import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


In [3]:
# Dataset location (Kaggle path)
DATASET_PATH = "/kaggle/input/towards-ds-20records/towards_data_science_100_articles.csv"
OUTPUT_CSV = "/kaggle/working/summarized_towards_ds.csv"

# Columns to use
TEXT_COL = "Content"
TITLE_COL = "Title"

# Model to use
MODEL_ID = "sshleifer/distilbart-cnn-12-6"

# Tokenization chunking settings
MAX_INPUT_TOKENS = 1024  # Model input limit
CHUNK_OVERLAP = 50       # Overlap for better continuity


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID).to(device)

if device.type == "cuda":
    model.half()  # Speeds up inference on GPU

model.eval()  # Set model to inference mode


Using device: cuda


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

2025-07-27 07:55:50.779758: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753602951.153295      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753602951.262266      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartSdpaAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
    

In [5]:
def chunk_text(text, tokenizer, max_tokens=1024, stride=50):
    tokens = tokenizer.encode(text, truncation=False)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk = tokens[start:end]
        chunks.append(tokenizer.decode(chunk, skip_special_tokens=True))
        start += max_tokens - stride
    return chunks


In [6]:
def generate_n_summaries(text, tokenizer, model, n=3):
    """
    Generates N different summaries for the given input text.
    
    Args:
        text (str): The full input text.
        tokenizer: Tokenizer for the model.
        model: The summarization model.
        n (int): Number of summaries to generate.
    
    Returns:
        List[str]: List of generated summaries.
    """
    device = model.device
    chunks = chunk_text(text, tokenizer)

    # Merge all chunk summaries into one large base input
    chunk_summaries = []
    for chunk in chunks:
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=1024).to(device)
        summary_ids = model.generate(
            **inputs,
            max_length=150,
            min_length=30,
            length_penalty=2.0,
            num_beams=4,
            early_stopping=True
        )
        summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        chunk_summaries.append(summary)
    
    combined_summary_text = " ".join(chunk_summaries)

    # Generate `n` variations of final summary using sampling
    inputs = tokenizer(combined_summary_text, return_tensors="pt", truncation=True, max_length=1024).to(device)
    
    final_summaries = []
    for _ in range(n):
        summary_ids = model.generate(
            **inputs,
            max_length=200,
            min_length=80,
            do_sample=True,              # Enables sampling for diverse outputs
            top_k=50,                    # Top-k sampling
            top_p=0.95,                  # Nucleus sampling
            temperature=1.0,             # Controls randomness
            no_repeat_ngram_size=3,     # Avoid repetition
            num_beams=1,                # No beam search for diversity
            early_stopping=True
        )
        summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        final_summaries.append(summary)

    return final_summaries


In [7]:
# Load dataset and drop rows with null content
df = pd.read_csv(DATASET_PATH)
df = df.dropna(subset=[TEXT_COL])

# Enable tqdm progress bar for Pandas
tqdm.pandas()

# Apply summarization
def summarize_row(content):
    summaries = generate_n_summaries(content, tokenizer, model, n=3)
    return pd.Series({
        "summary_1": summaries[0],
        "summary_2": summaries[1],
        "summary_3": summaries[2]
    })

summary_df = df[TEXT_COL].progress_apply(summarize_row)
df = pd.concat([df, summary_df], axis=1)



  0%|          | 0/20 [00:00<?, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (1454 > 1024). Running this sequence through the model will result in indexing errors
The following generation flags are not valid and may be ignored: ['early_stopping', 'length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['early_stopping', 'length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['early_stopping', 'length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

 10%|█         | 2/20 [00:06<00:57,  3.21s/it]The following generation flags are not valid and may be ignored: ['early_stopping', 'length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['early_stopping', 'length_penalty']. Set `TR

In [8]:
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Saved with 3 summaries per record → {OUTPUT_CSV}")


✅ Saved with 3 summaries per record → /kaggle/working/summarized_towards_ds.csv
